# DX 704 Week 8 Project

This homework will modify a simulator controlling a small vehicle to implement tabular q-learning.
You will first test your code with random and greedy-epsilon policies, then tweak your own training method for a more optimal policy.

The full project description and a template notebook are available on GitHub: [Project 8 Materials](https://github.com/bu-cds-dx704/dx704-project-08).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Rover Simulator

The following Python class implements a simulation of a simple vehicle with integer x,y coordinates facing in one of 8 possible directions.


In [5]:
# DO NOT CHANGE

import random

class RoverSimulator(object):
    DIRECTIONS = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    def __init__(self, resolution):
        self.resolution = resolution
        self.terminal_state = self.construct_state(resolution // 2, resolution // 2, 0)

        self.initial_states = []
        for initial_x in (0, resolution // 2, resolution - 1):
            for initial_y in (0, resolution // 2, resolution - 1):
                for initial_direction in range(8):
                    initial_state = self.construct_state(initial_x, initial_y, initial_direction)
                    if initial_state != self.terminal_state:
                        self.initial_states.append(initial_state)

    def construct_state(self, x, y, direction):
        assert 0 <= x < self.resolution
        assert 0 <= y < self.resolution
        assert 0 <= direction < 8

        state = (y * self.resolution + x) * 8 + direction
        assert self.decode_state(state) == (x, y, direction)
        return state

    def decode_state(self, state):
        direction = state % 8
        x = (state // 8) % self.resolution
        y = state // (8 * self.resolution)

        return (x, y, direction)

    def get_actions(self, state):
        return [-1, 0, 1]

    def get_next_reward_state(self, curr_state, curr_action):
        if curr_state == self.terminal_state:
            # no rewards or changes from terminal state
            return (0, curr_state)

        (curr_x, curr_y, curr_direction) = self.decode_state(curr_state)
        (curr_dx, curr_dy) = self.DIRECTIONS[curr_direction]

        assert self.construct_state(curr_x, curr_y, curr_direction) == curr_state

        assert curr_action in (-1, 0, 1)

        next_x = min(max(0, curr_x + curr_dx), self.resolution - 1)
        next_y = min(max(0, curr_y + curr_dy), self.resolution - 1)
        next_direction = (curr_direction + curr_action) % 8

        next_state = self.construct_state(next_x, next_y, next_direction)
        next_reward = 1 if next_state == self.terminal_state else 0

        return (next_reward, next_state)

    def rollout_policy(self, policy_func, max_steps=1000):
        curr_state = self.sample_initial_state()
        for i in range(max_steps):
            curr_action = policy_func(curr_state, self.get_actions(curr_state))
            (next_reward, next_state) = self.get_next_reward_state(curr_state, curr_action)
            yield (curr_state, curr_action, next_reward, next_state)
            curr_state = next_state

    def sample_initial_state(self):
        return random.choice(self.initial_states)

In [7]:
simulator = RoverSimulator(16)
initial_sample = simulator.sample_initial_state()
print("INITIAL SAMPLE", initial_sample)

INITIAL SAMPLE 1091


## Part 1: Implement a Random Policy

Random policies are often used to test simulators and start initial exploration.
Implement a random policy for these simulators.

In [13]:
def random_policy(state, actions):
    return random.choice(actions)

Use the code below to test your random policy.
Then modify it to save the results in "log-random.tsv" with the columns curr_state, curr_action, next_reward and next_state.

In [14]:
import csv

with open('log-random.tsv', 'w', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['curr_state', 'curr_action', 'next_reward', 'next_state'])
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=32):
        writer.writerow([curr_state, curr_action, next_reward, next_state])
        print("CURR STATE", curr_state, "ACTION", curr_action, "NEXT REWARD", next_reward, "NEXT STATE", next_state)

CURR STATE 1030 ACTION 1 NEXT REWARD 0 NEXT STATE 1031
CURR STATE 1031 ACTION 1 NEXT REWARD 0 NEXT STATE 1152
CURR STATE 1152 ACTION -1 NEXT REWARD 0 NEXT STATE 1287
CURR STATE 1287 ACTION 0 NEXT REWARD 0 NEXT STATE 1415
CURR STATE 1415 ACTION 1 NEXT REWARD 0 NEXT STATE 1536
CURR STATE 1536 ACTION 0 NEXT REWARD 0 NEXT STATE 1664
CURR STATE 1664 ACTION 1 NEXT REWARD 0 NEXT STATE 1793
CURR STATE 1793 ACTION 0 NEXT REWARD 0 NEXT STATE 1929
CURR STATE 1929 ACTION 0 NEXT REWARD 0 NEXT STATE 1937
CURR STATE 1937 ACTION -1 NEXT REWARD 0 NEXT STATE 1944
CURR STATE 1944 ACTION 0 NEXT REWARD 0 NEXT STATE 1944
CURR STATE 1944 ACTION -1 NEXT REWARD 0 NEXT STATE 1951
CURR STATE 1951 ACTION -1 NEXT REWARD 0 NEXT STATE 1942
CURR STATE 1942 ACTION 0 NEXT REWARD 0 NEXT STATE 1934
CURR STATE 1934 ACTION 1 NEXT REWARD 0 NEXT STATE 1927
CURR STATE 1927 ACTION 0 NEXT REWARD 0 NEXT STATE 1927
CURR STATE 1927 ACTION -1 NEXT REWARD 0 NEXT STATE 1926
CURR STATE 1926 ACTION 0 NEXT REWARD 0 NEXT STATE 1926
CURR 

Submit "log-random.tsv" in Gradescope.

## Part 2: Implement Q-Learning with Random Policy

The code below runs 32 random rollouts of 1024 steps using your random policy.
Modify the rollout code to implement Q-Learning.
Just implement one learning update for each sampled state-action in the simulation.
Use $\alpha=1$ and $\gamma=0.9$ since the simulator is deterministic and there is a sink where the rewards stop.




In [17]:
random.seed(0)
q_values = {}
q_random_rows = []

for episode in range(32):
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(random_policy, max_steps=1024):
        old_value = q_values.get((curr_state, curr_action), 0.0)
        next_values = [
            q_values.get((next_state, next_action), 0.0)
            for next_action in simulator.get_actions(next_state)
        ]
        new_value = next_reward + 0.9 * max(next_values)
        q_values[(curr_state, curr_action)] = new_value
        q_random_rows.append((curr_state, curr_action, next_reward, next_state, old_value, new_value))

print(f"Logged {len(q_random_rows)} Q-learning updates")
print(f"Terminal rewards observed: {sum(row[2] for row in q_random_rows)}")

Logged 32768 Q-learning updates
Terminal rewards observed: 5


Save each step in the simulator in a file "q-random.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

In [18]:
import csv

with open('q-random.tsv', 'w', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['curr_state', 'curr_action', 'next_reward', 'next_state', 'old_value', 'new_value'])
    writer.writerows(q_random_rows)

print(f"Wrote {len(q_random_rows)} rows to q-random.tsv")

Wrote 32768 rows to q-random.tsv


Submit "q-random.tsv" in Gradescope.

## Part 3: Implement Epsilon-Greedy Policy

Implement an epsilon-greedy policy that picks the optimal policy based on your q-values so far 75% of the time, and picks a random action 25% of the time.
This is a high epsilon value, but the environment is deterministic, so it will benefit from more exploration.

In [19]:
# hard-code epsilon=0.25. this is high but the environment is deterministic.
def epsilon_greedy_policy(state, actions):
    if random.random() < 0.25:
        return random.choice(actions)
    state_values = [q_values.get((state, action), 0.0) for action in actions]
    return actions[state_values.index(max(state_values))]

Combine your epsilon-greedy policy with q-learning below and save the observations and updates in "q-greedy.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

Hint: make sure to reset your q-learning state before running the simulation below so that the learning process is recorded from the beginning.

In [ ]:
random.seed(1)
q_values = {}

with open('q-greedy.tsv', 'w', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['curr_state', 'curr_action', 'next_reward', 'next_state', 'old_value', 'new_value'])
    row_count = 0
    for episode in range(32):
        for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):
            old_value = q_values.get((curr_state, curr_action), 0.0)
            next_values = [
                q_values.get((next_state, next_action), 0.0)
                for next_action in simulator.get_actions(next_state)
            ]
            new_value = next_reward + 0.9 * max(next_values)
            q_values[(curr_state, curr_action)] = new_value
            writer.writerow([curr_state, curr_action, next_reward, next_state, old_value, new_value])
            row_count += 1

print(f"Wrote {row_count} rows to q-greedy.tsv")

Wrote 14032 rows to q-greedy.tsv


Submit "q-greedy.tsv" in Gradescope.

## Part 4: Extract Policy from Q-Values

Using your final q-values from the previous simulation, extract a policy picking the best actions according to those q-values.
Save the policy in a file "policy-greedy.tsv" with columns state and action.

In [21]:
policy_rows = []
for state in sorted({state for (state, action) in q_values}):
    best_action = max(
        (-1, 0, 1),
        key=lambda action: q_values.get((state, action), 0.0)
    )
    policy_rows.append((state, best_action))

with open('policy-greedy.tsv', 'w', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['state', 'action'])
    writer.writerows(policy_rows)

print(f"Wrote policy for {len(policy_rows)} states to policy-greedy.tsv")

Wrote policy for 1674 states to policy-greedy.tsv


Submit "policy-greedy.tsv" in Gradescope.

## Part 5: Implement Large Policy

Train a more optimal policy using q-learning.
Save the policy in a file "policy-optimal.tsv" with columns state and action.

Hint: this policy will be graded on its performance compared to optimal for each of the initial states.
**You will get full credit if the average value of your policy for the initial states is within 20% of optimal.**
Make sure that your policy has coverage of all the initial states, and does not take actions leading to states not included in your policy.
You will have to run several rollouts to get coverage of all the initial states, and the provided loops for parts 2 and 3 only consist of one rollout each.

Hint: this environment only gives one non-zero reward per episode, so you may want to cut off rollouts for speed once they get that reward.
But make sure you update the q-values first!

In [22]:
# Train a more optimal policy with many episodes covering all initial states.
random.seed(42)
q_values_optimal = {}

def optimal_training_policy(state, actions):
    epsilon = 0.25
    if random.random() < epsilon:
        return random.choice(actions)
    state_values = [q_values_optimal.get((state, action), 0.0) for action in actions]
    return actions[state_values.index(max(state_values))]

# Run many episodes from each initial state for coverage.
for episode in range(2000):
    start_state = simulator.initial_states[episode % len(simulator.initial_states)]
    curr_state = start_state
    for step in range(1024):
        actions = simulator.get_actions(curr_state)
        curr_action = optimal_training_policy(curr_state, actions)
        next_reward, next_state = simulator.get_next_reward_state(curr_state, curr_action)
        old_value = q_values_optimal.get((curr_state, curr_action), 0.0)
        next_values = [q_values_optimal.get((next_state, a), 0.0) for a in simulator.get_actions(next_state)]
        new_value = next_reward + 0.9 * max(next_values)
        q_values_optimal[(curr_state, curr_action)] = new_value
        curr_state = next_state
        if next_reward > 0:
            break

# Extract the optimal policy from the final Q-values.
optimal_policy = {}
for state in sorted({state for (state, action) in q_values_optimal}):
    best_action = max(
        (-1, 0, 1),
        key=lambda action: q_values_optimal.get((state, action), 0.0)
    )
    optimal_policy[state] = best_action

# Evaluate the policy from every initial state.
def policy_function(state, actions):
    return optimal_policy.get(state, 0)

initial_state_values = []
for initial_state in simulator.initial_states:
    total_reward = 0.0
    discount = 1.0
    for (curr_state, curr_action, next_reward, next_state) in simulator.rollout_policy(policy_function, max_steps=1024):
        total_reward += discount * next_reward
        discount *= 0.9
        if next_reward > 0:
            break
    initial_state_values.append(total_reward)

average_value = sum(initial_state_values) / len(initial_state_values)
print(f"Trained on {2000} episodes covering all {len(simulator.initial_states)} initial states")
print(f"Policy covers {len(optimal_policy)} states")
print(f"Average initial-state value: {average_value:.4f}")
print(f"States with positive value: {sum(1 for v in initial_state_values if v > 0)}/{len(initial_state_values)}")

# Save the optimal policy.
with open('policy-optimal.tsv', 'w', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['state', 'action'])
    writer.writerows(sorted(optimal_policy.items()))

print(f"Wrote {len(optimal_policy)} states to policy-optimal.tsv")

Trained on 2000 episodes covering all 71 initial states
Policy covers 1910 states
Average initial-state value: 0.3342
States with positive value: 71/71
Wrote 1910 states to policy-optimal.tsv


Submit "policy-optimal.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

### Acknowledgement

No other people provided assistance. I used Python's built-in `random` and `csv` modules, plus the supplied RoverSimulator class, to implement tabular Q-learning. GitHub Copilot Chat in Visual Studio Code assisted with interpreting the assignment, implementing and debugging the Q-learning code, and validating the output files.

[Copilot Chat transcript](file:///Users/christopherabbas/Library/Application%20Support/Code/User/workspaceStorage/d690c5667068afdbf94ff162bc48cae2/GitHub.copilot-chat/transcripts/e7d90d9c-3572-4733-8668-e31c0026bd5e.jsonl)